# 第 2 周额外周末练习 —— 技术问答导师原型

## 练习目标（理念）

把第 1 周的「技术问题解释器」升级成可用原型，综合运用第 2 周知识点：

- **Gradio UI**：聊天界面，而不是只在笔记本里 `print`
- **流式（streaming）**：边生成边显示
- **System Prompt**：注入「LLM 工程导师」专业人设
- **多模型切换**：OpenRouter / Groq / Anthropic / Gemini / 本地 Ollama
- **奖励**：工具调用（内置知识库）+ 本地 TTS（`pyttsx3`）

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多提供商 OpenAI 兼容客户端 | `base_url` 指向各家 API |
| System Prompt | `system_prompt` 定导师风格 |
| Tool / Function Calling | `knowledge_base_lookup` + `_run_tool_loop` |
| Streaming | `chat_stream` 里 `stream=True` + `yield` |
| Gradio ChatInterface | 下拉选模型 + examples |

## 怎么跑

1. 准备 `.env`：按需设置 `OR_API_KEY`、`GROQ_API_KEY`、`ANTHROPIC_API_KEY`、`GEMINI_API_KEY`
2. 若用本地模型：确保 Ollama 在 `http://localhost:11434`，并已拉取 `llama3.2:3b`
3. 从上到下运行单元格；Gradio 格会 `launch()` 打开界面


In [ ]:
# ========== 导入 + 多提供商客户端（OpenAI 兼容接口） ==========
# 第 2 周 Day 1：同一套 OpenAI SDK，换 base_url / api_key 即可连不同后端

# 导入标准库 os：读环境变量（Environment Variables）里的各家 API Key
import os
# 导入标准库 json：工具参数字符串 ↔ Python 字典
import json
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：云端与本地兼容端点共用
from openai import OpenAI
# 导入 gradio：快速搭聊天 UI（ChatInterface）
import gradio as gr

# override=True：.env 中的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)

# 分别读取各提供商密钥（变量名与 .env 键保持一致，勿改）
or_api_key      = os.getenv('OR_API_KEY')
groq_api_key    = os.getenv('GROQ_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
gemini_api_key  = os.getenv('GEMINI_API_KEY')

# 客户端——全部通过 OpenAI 兼容接口（第 2 周，第 1 天技术）
# OpenRouter：聚合多家模型的网关
or_client       = OpenAI(api_key=or_api_key,       base_url="https://openrouter.ai/api/v1")
# Groq：低延迟推理（常用于 Llama 等开源权重托管）
groq_client     = OpenAI(api_key=groq_api_key,     base_url="https://api.groq.com/openai/v1")
# Anthropic：官方 OpenAI 兼容端点（URL / 密钥逻辑保持原样）
anthropic_client= OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1")
# Google Gemini：OpenAI 兼容 beta 路径
gemini_client   = OpenAI(api_key=gemini_api_key,   base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
# 本地 Ollama：api_key 占位字符串即可（SDK 要求非空）；base_url 指向本机 /v1
ollama_client   = OpenAI(api_key="ollama",          base_url="http://localhost:11434/v1")

# 打印初始化状态（文案保持英文原样，便于对照运行输出）
print("All clients initialised!")
# 逐家检查密钥是否存在：有则只显示前 8 位，避免泄露完整 key
for name, key in [("OpenRouter", or_api_key), ("Groq", groq_api_key),
                  ("Anthropic", anthropic_api_key), ("Gemini", gemini_api_key)]:
    print(f"  {name}: {'\u2713 ' + key[:8] if key else '\u2717 not set'}")


In [ ]:
# ========== System Prompt + 可用模型表 ==========
# Day 3：用 system 注入「技术导师」专业知识；Day 1：按密钥动态拼 MODELS 下拉选项

# 系统提示——增加了深厚的技术专业知识（第2周，第3天技术）
# 发给模型的英文正文必须保留：改译会改变回答风格/行为
system_prompt = """You are an expert technical tutor specialising in LLM Engineering and AI development.
Your students are software engineers learning to build AI-powered applications.

When answering:
- Provide clear, structured explanations with concrete code examples in markdown code blocks
- Draw simple ASCII diagrams for complex architectures when helpful
- Explain *why* things work, not just *what* they do
- Relate concepts to real-world LLM/AI engineering use cases
- Suggest related topics the student might want to explore next

You have access to a knowledge-base tool -- use it when a student asks about a recognised topic
(tokenization, embeddings, attention, RAG, fine-tuning, prompting, tools/function-calling).

Always respond in markdown."""

# --- 可用模型（第 2 周，第 1 天）---
# 首先优先考虑无信贷/低成本选项。
# MODELS：展示名 → {client 别名, model id}；后面 Dropdown 直接用 keys
MODELS = {}

# 通过 Groq 的免费/低成本云（如果设置了密钥）
if groq_api_key:
    # 键名是 UI 展示文案；model 字符串必须与提供商目录一致
    MODELS["Llama 3.3 70B (Groq -- fast)"] = {"client": "groq", "model": "llama-3.3-70b-versatile"}
    MODELS["GPT-OSS 20B (Groq)"] = {"client": "groq", "model": "openai/gpt-oss-20b"}

# 通过 Ollama 的本地无信用模式（始终包含在内）
MODELS["Llama 3.2 3B (Ollama -- local)"] = {"client": "ollama", "model": "llama3.2:3b"}

# 可选的付费/附加提供商（仅在密钥存在时显示）
if or_api_key:
    MODELS["GPT-4o-mini (OpenRouter)"] = {"client": "or", "model": "openai/gpt-4o-mini"}
if anthropic_api_key:
    MODELS["Claude Haiku 4.5 (Anthropic)"] = {"client": "anthropic", "model": "claude-haiku-4-5"}
if gemini_api_key:
    MODELS["Gemini 2.0 Flash (Google)"] = {"client": "gemini", "model": "gemini-2.0-flash"}

# 别名 → 真实客户端对象：get_client 时做二次查找
_client_map = {
    "or": or_client,
    "anthropic": anthropic_client,
    "gemini": gemini_client,
    "groq": groq_client,
    "ollama": ollama_client,
}


def get_client(model_name: str):
    # 按 UI 选中的展示名取出 client 实例与真正的 model id
    cfg = MODELS[model_name]
    return _client_map[cfg["client"]], cfg["model"]


# 列出当前可用模型，方便确认密钥是否生效
print("Models available:")
for name in MODELS:
    print(f"  - {name}")


In [ ]:
# ========== 奖励工具：小型内置知识库（Function Calling） ==========
# Day 4：把 KB 查表封装成工具，让模型在相关主题上主动 call function

# 主题 → 简短英文释义（工具返回给模型的内容保持原样）
KB = {
    "tokenization": "Tokenization converts text into tokens that the model can process. Modern LLM tokenizers use subword/BPE-style vocabularies to balance efficiency and coverage.",
    "embeddings": "Embeddings map text to dense vectors where semantic similarity is represented by geometric closeness. They power retrieval, clustering, and semantic search.",
    "attention": "Attention lets each token dynamically weight other tokens, enabling context-aware representations and strong sequence modeling.",
    "rag": "RAG (Retrieval-Augmented Generation) retrieves external context at runtime and injects it into prompts to improve factuality and domain coverage.",
    "fine-tuning": "Fine-tuning updates model weights on curated data for style/domain adaptation, while prompting keeps base weights unchanged.",
    "prompting": "Prompting is the art of structuring instructions/context/examples so the model produces better outputs without retraining.",
    "tools": "Tool/function calling lets the model request external functions (APIs, DB lookups, calculators), then continue reasoning with returned results.",
}


def knowledge_base_lookup(topic: str) -> str:
    # 规范化查询：去空白 + 小写，便于模糊匹配
    query = (topic or "").strip().lower()
    if not query:
        # 空查询时的提示文案保持原样（可能被模型读到）
        return "Please provide a topic to look up."

    # 模糊匹配，以便用户可以输入自然变体（双向 substring）
    for key, value in KB.items():
        if query in key or key in query:
            return f"Topic: {key}\n\n{value}"

    # 未命中：列出可用主题，引导用户重试
    topics = ", ".join(sorted(KB.keys()))
    return f"I don't have that topic yet. Try one of: {topics}"


# OpenAI tools schema：告诉模型「有什么函数、参数长什么样」
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "knowledge_base_lookup",
            "description": "Look up a short explanation for a core LLM engineering concept.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The concept to retrieve (e.g., tokenization, embeddings, rag, prompting)",
                    }
                },
                "required": ["topic"],
            },
        },
    }
]

print("Knowledge-base tool ready!")


In [ ]:
# ========== 核心聊天：工具循环 + 流式最终回答 ==========
# 第 4 天模式：在模型要求工具时继续循环；第 5 天：最终答案 stream=True 逐 token yield


def _serialise_tool_call(tc):
    # 把 SDK 的 tool_call 对象转成可放进 messages 的 dict
    if hasattr(tc, "model_dump"):
        # Pydantic v2：优先用官方序列化
        return tc.model_dump()
    # 兼容旧对象：手工拼 OpenAI 期望的结构
    return {
        "id": tc.id,
        "type": "function",
        "function": {
            "name": tc.function.name,
            "arguments": tc.function.arguments,
        },
    }


def _history_to_messages(history):
    # Gradio 历史可能是 messages 字典列表，也可能是旧版 [user, assistant] 二元组
    messages = []
    for turn in history or []:
        # 新版 type="messages"：直接带 role/content
        if isinstance(turn, dict) and "role" in turn and "content" in turn:
            messages.append({"role": turn["role"], "content": turn["content"]})
            continue
        # 旧版二元组兼容
        if isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_msg, assistant_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    return messages


def _run_tool_loop(client, model_id: str, messages: list, max_rounds: int = 4):
    """Resolve one or more tool calls before final answer stream."""
    # working：可变消息列表，工具结果会不断 append
    working = list(messages)

    for _ in range(max_rounds):
        # 非流式探测：看模型是否要调工具（stream=False 才能读 finish_reason / tool_calls）
        response = client.chat.completions.create(
            model=model_id,
            messages=working,
            tools=TOOLS,
            tool_choice="auto",
            stream=False,
            temperature=0.3,
        )

        choice = response.choices[0]
        msg = choice.message
        finish_reason = getattr(choice, "finish_reason", None)
        tool_calls = getattr(msg, "tool_calls", None) or []

        # 没有工具请求 -> 我们已准备好进行最终的流式生成
        if finish_reason != "tool_calls" or not tool_calls:
            if msg.content:
                # 若探测轮已带正文，先写入历史，供后续 stream 承接上下文
                working.append({"role": "assistant", "content": msg.content})
            return working

        # 先把「助手发起的 tool_calls」写入对话（协议要求）
        working.append(
            {
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [_serialise_tool_call(tc) for tc in tool_calls],
            }
        )

        # 逐个执行工具，结果以 role=tool 回灌
        for tc in tool_calls:
            fn_name = tc.function.name
            # arguments 是 JSON 字符串
            args = json.loads(tc.function.arguments or "{}")

            if fn_name == "knowledge_base_lookup":
                result = knowledge_base_lookup(args.get("topic", ""))
            else:
                result = f"Unknown tool: {fn_name}"

            working.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": fn_name,
                    "content": result,
                }
            )

    # 如果模型不断重复调用工具，则安全回退（文案保持原样）
    working.append(
        {
            "role": "assistant",
            "content": "I reached the tool-call limit for this turn. Please refine your question and try again.",
        }
    )
    return working


def chat_stream(message, history, model_name):
    # Gradio ChatInterface 回调：message 本轮用户输入；history 既往；model_name 来自 Dropdown
    try:
        # 解析所选展示名 → (client, model_id)
        client, model_id = get_client(model_name)

        # 组装 messages：system + 历史 + 本轮 user
        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(_history_to_messages(history))
        messages.append({"role": "user", "content": message})

        # 先跑工具循环，拿到「已含 tool 结果」的消息列表
        prepared_messages = _run_tool_loop(client, model_id, messages)

        # 最终回答：开启流式，边收边 yield 给 Gradio
        stream = client.chat.completions.create(
            model=model_id,
            messages=prepared_messages,
            stream=True,
            temperature=0.3,
        )

        partial = ""
        for chunk in stream:
            delta = chunk.choices[0].delta
            # 有的 chunk 只有 role/空 content，用 or "" 兜底
            token = getattr(delta, "content", None) or ""
            partial += token
            # 每次 yield 完整迄今文本（ChatInterface 流式约定）
            yield partial

    except Exception as e:
        # 错误也 yield 出去，UI 可见（前缀 Error: 保持原样）
        yield f"Error: {e}"


print("Streaming + tool loop ready!")


In [ ]:
# ========== Gradio UI：ChatInterface + 模型下拉 + 示例问题 ==========
# Day 2 UI + Day 1 切换器 + Day 5 流式（fn 即上面的 chat_stream）

# 下拉框：选项来自 MODELS 的展示名；默认选第一个可用模型
model_dropdown = gr.Dropdown(
    choices=list(MODELS.keys()),
    value=list(MODELS.keys())[0],
    label="Choose model/provider",
)

# 示例问题（英文保持原样：会直接作为 user message 发送）
examples = [
    ["Explain retrieval augmented generation (RAG) with an architecture sketch."],
    ["When should I use fine-tuning instead of prompt engineering?"],
    ["Use the knowledge base tool to teach me tokenization."],
    ["Give me a practical checklist for production LLM observability."],
]

# ChatInterface：type="messages" 与 _history_to_messages 的 dict 分支对齐
ui = gr.ChatInterface(
    fn=chat_stream,
    type="messages",
    # 额外输入：把模型选择传给 chat_stream 的第三个参数
    additional_inputs=[model_dropdown],
    title="Technical Q/A Tutor",
    description="Applies Day1-Day5 concepts: multi-provider routing, structured system prompting, tool-calling loop, and streaming answers.",
    examples=examples,
)

# queue()：支持流式/排队；launch() 启动本地 Web UI
ui.queue().launch()


In [ ]:
# ========== 可选奖励：本地离线 TTS（pyttsx3，不消耗云端积分） ==========
# Day 5 音频方向：把文本存成 wav；失败时返回 (None, 原因字符串)

# 导入 tempfile：生成不冲突的临时文件路径
import tempfile


def text_to_speech_file(text: str):
    """Generate local speech audio if pyttsx3 is available."""
    try:
        # 延迟导入：没装 pyttsx3 时不让整本笔记本挂掉
        import pyttsx3
    except Exception:
        # 安装提示文案保持原样
        return None, "pyttsx3 not installed. Run: uv add pyttsx3"

    try:
        # 初始化本机 TTS 引擎
        engine = pyttsx3.init()
        # delete=False：关闭句柄后文件仍保留，供后续播放
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        path = tmp.name
        tmp.close()

        # 截断到 2000 字符，避免超长朗读；save_to_file 异步写入任务队列
        engine.save_to_file((text or "")[:2000], path)
        # 阻塞直到合成完成
        engine.runAndWait()
        return path, "Audio created locally (offline)."
    except Exception as e:
        return None, f"Local audio generation failed: {e}"


# 用法提示（字符串保持原样）
print("Optional local audio helper ready. Example:")
print("audio_path, status = text_to_speech_file('Hello from Week 2 Day 5')")


In [ ]:
# ========== 试跑本地 TTS：示例句 1 ==========
# 返回 (wav 路径或 None, 状态说明)；需已安装 pyttsx3

audio_path, status = text_to_speech_file('Hello from Week 2 Day 5')


In [ ]:
# ========== 试跑本地 TTS：示例句 2 ==========
# 换一句问候，确认引擎可重复调用

audio_path, status = text_to_speech_file('Hello, My name is Ed, nice to meet you')
